# View one slice at full resolution

Opens one section in napari at the scanner's full resolution (0.325 µm/px), with each stain as its own layer and the saved T1–T6 ROIs on top. Zoom and pan freely: napari loads tiles on demand, so the whole 7 GB image is never in memory.

The first time a section is opened, it's exported from the `.vsi` at full resolution. That takes **~7 min and ~7 GB of disk**. It's cached in `data/processed/histology/Mouse_NN/full/`, which is git-ignored; the last cell deletes it.

Kernel: **Python 3.12 (fus-research .venv)**.

In [1]:
%gui qt
from pathlib import Path
import re

import dask.array as da
import napari
import numpy as np
import pandas as pd
import tifffile
import zarr

from fus import histology, orientation, rois

ROOT = histology.REPO_ROOT

MOUSE = 2
SECTION = "slide04_s3"   # e.g. section_s4 (Mouse 1) or slide01_s3 (Mouse 2)

In [2]:
# Find the .vsi and series for this section; export at full resolution if not cached.
raw = ROOT / f"data/histology/Mouse_{MOUSE:02d}"
prefix, series = re.fullmatch(r"(.+)_s(\d+)", SECTION).groups()
if prefix == "section":
    vsi = raw / "Image.vsi"
else:
    vsi = next(raw.glob(f"*Slide_{int(prefix.removeprefix('slide')):02d}.vsi"))
full = ROOT / f"data/processed/histology/Mouse_{MOUSE:02d}/full/{SECTION}.ome.tif"
if not full.exists():
    print(f"exporting {vsi.name} series {series} at full resolution (~7 min)...")
    histology.export_sections(vsi, full.parent, downsample=1, prefix=prefix, series=int(series))
print(full.relative_to(ROOT), f"{full.stat().st_size / 1e9:.1f} GB")

data/processed/histology/Mouse_02/full/slide04_s3.ome.tif 7.0 GB


In [3]:
# Open lazily: one multiscale layer per stain, in µm.
with tifffile.TiffFile(full) as tf:
    meta = tf.ome_metadata
px_um = float(meta.split('PhysicalSizeX="', 1)[1].split('"', 1)[0])
channels = [c.split('Name="', 1)[1].split('"', 1)[0] for c in meta.split("<Channel ")[1:]]
group = zarr.open(tifffile.imread(full, aszarr=True), mode="r")
pyramid = [da.from_zarr(group[k]) for k in sorted(group.keys(), key=int)]

STYLE = {  # colormap, visible, opacity
    "CY5":   ("gray", True, 0.4),      # NeuN
    "TRITC": ("green", True, 1.0),     # anti-GFP stain: the delivery measure
    "FITC":  ("yellow", False, 1.0),   # native GFP
    "DAPI":  ("blue", False, 1.0),     # nuclei
}
NAMES = {"CY5": "NeuN (CY5)", "TRITC": "GFP stain (TRITC)", "FITC": "native GFP (FITC)", "DAPI": "DAPI"}

viewer = napari.Viewer(title=f"Mouse {MOUSE} — {SECTION}")
for i, ch in enumerate(channels):
    cmap, visible, opacity = STYLE.get(ch, ("gray", False, 1.0))
    small = pyramid[-1][i].compute()
    viewer.add_image([level[i] for level in pyramid], multiscale=True, name=NAMES.get(ch, ch),
                     colormap=cmap, blending="additive", visible=visible, opacity=opacity,
                     contrast_limits=tuple(np.percentile(small, [1, 99.8])), scale=(px_um, px_um),
                     units="um")

# ROIs: finetuned shapes if saved, else the template from the orientation clicks.
ds4 = ROOT / f"data/processed/histology/Mouse_{MOUSE:02d}/ds4/{SECTION}.ome.tif"
ds4_um = rois.export_pixel_um(ds4)
shapes = rois.read_locations().get((MOUSE, SECTION))
source = "finetuned"
if shapes is None:
    a = orientation.read_sheet()[(MOUSE, SECTION)]
    mirrored = orientation.KNOWN_MIRRORED.get((MOUSE, SECTION))
    if mirrored is None:
        mirrored = orientation.is_mirrored(a, orientation.plus_x_side(orientation.read_sheet()))
    shapes = rois.from_template(a, ds4_um, mirrored)
    source = "template"
slots = pd.read_csv(ROOT / f"results/histology/mouse{MOUSE:02d}_template_slots.csv")
cov = slots[(slots.section == SECTION) & (slots.channel == "TRITC")].set_index("target").coverage
targets = sorted(shapes)
layer = viewer.add_shapes([shapes[t].corners() for t in targets], shape_type="ellipse",
                          name=f"ROIs ({source})", edge_color="yellow", face_color="transparent",
                          edge_width=8, scale=(ds4_um, ds4_um), units="um")
layer.features = pd.DataFrame({"label": [f"T{t} {cov.get(t, np.nan):.0%}" for t in targets]})
layer.text = {"string": "{label}", "color": "yellow", "size": 14}
layer.editable = False

viewer.canvas.overlays.scale_bar.visible = True
viewer.reset_view()

Toggle stains with the eye icons in the layer list. Contrast is set from the whole slice; adjust it per layer in the layer controls. ROI labels show TRITC coverage from the last measurement.

In [ ]:
# Free the disk space when done (the export is re-made on the next open).
# full.unlink()